# Самостоятельная работа 04. Деревья решений

Цель: вручную посчитать неоднородность Джини и критерий CART для вопроса дерева, проверить расчёт деревом scikit-learn, найти лучший вопрос перебором и увидеть, как регуляризация лечит переобучение дерева, а удаление одного объекта меняет его корень.

Правила:

- защищать можно любое число заданий — от одного до всех;
- уйти с занятия раньше времени можно только после защиты хотя бы одного задания;
- баллы ставятся только за защищённые задания: базовое — 1 балл, продвинутое — 3 балла; выполненное, но не защищённое задание даёт 0;
- на защите нужно объяснить любую строку своего кода и при преподавателе изменить его;
- ИИ-помощником можно пользоваться точечно — вспомнить функцию, разобраться с ошибкой; код, который вы не можете объяснить, не засчитывается;
- перед защитой перезапустите ядро (Kernel → Restart) и выполните сверху вниз подготовку данных и защищаемые задания — их самопроверки должны пройти.

Материалы:

- лекция 06 «Деревья решений»;
- Орельен Жерон, «Прикладное машинное обучение с помощью Scikit-Learn и TensorFlow», гл. 6, стр. 223–236;
- для задания 1 нужны листок бумаги, ручка и калькулятор — листок понадобится на защите.

| № | Что сделать | Тип | Баллы |
|---|---|---|---|
| 1 | Посчитать на листке Джини и $J$ своего варианта | базовое | 1 |
| 2 | Проверить листок деревом scikit-learn | базовое | 1 |
| 3 | Перебрать все варианты и найти лучший вопрос | базовое | 1 |
| 4 | Переобучение, регуляризация и неустойчивость дерева | продвинутое | 3 |

In [ ]:
import sys

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from sklearn.tree import DecisionTreeClassifier, plot_tree

sys.version, np.__version__, pd.__version__, sklearn.__version__, matplotlib.__version__

## Подготовка данных

Ячейки ниже готовы — просто выполните их. Уличная камера с детектором движения записала 15 сработок. Тревога нужна, только когда в кадре человек; остальные сработки ложные: ветки на ветру, насекомое на объективе, блики фар, собака. У каждой сработки:

- `duration` — длительность движения, с;
- `area` — площадь области движения, % кадра;
- `source` — что вызвало сработку; `is_person` — класс: 1 — человек, 0 — ложная сработка.

Переменные: `events` — таблица сработок, индекс — номер сработки; `FEATURES` — имена признаков; `X_events`, `y_events` — признаки и классы для scikit-learn.

In [ ]:
events = pd.DataFrame(
    data=[
        (9, 4, 'человек', 1),
        (27, 13, 'ветки', 0),
        (2, 30, 'насекомое', 0),
        (13, 11, 'человек', 1),
        (7, 3, 'собака', 0),
        (34, 8, 'ветки', 0),
        (6, 5, 'человек', 1),
        (3, 24, 'блик фар', 0),
        (15, 6, 'человек', 1),
        (40, 15, 'ветки', 0),
        (1, 38, 'насекомое', 0),
        (11, 7, 'человек', 1),
        (22, 10, 'ветки', 0),
        (4, 33, 'блик фар', 0),
        (8, 9, 'человек', 1),
    ],
    columns=['duration', 'area', 'source', 'is_person'],
    index=pd.RangeIndex(1, 16, name='event'),
)
FEATURES = ['duration', 'area']
X_events = events[FEATURES]
y_events = events['is_person']

events

In [ ]:
fig, axis = plt.subplots(figsize=(7, 5))

for label, name, marker in ((1, 'человек', 'o'), (0, 'ложная сработка', 's')):
    group = events[events['is_person'] == label]
    axis.scatter(group['duration'], group['area'], marker=marker, s=50, label=name)

for number, row in events.iterrows():
    axis.annotate(str(number), (row['duration'], row['area']), xytext=(4, 4), textcoords='offset points')

axis.set_xlabel('Длительность движения, с')
axis.set_ylabel('Площадь области движения, % кадра')
axis.legend()
axis.grid(alpha=0.3)
plt.show()

Таблица вариантов `variants`: каждый вариант — вопрос «признак ≤ порог?». Порог — середина между соседними значениями признака, как на слайде «Перебираем пороги». Четыре крайних порога отрезают одну сработку — такой вопрос слишком прост, они исключены; остаётся 24 варианта.

**Ваш вариант — ваш порядковый номер в списке группы; если номер больше 24, вычтите из него 24.**

In [ ]:
variant_rows = []

for feature in FEATURES:
    values = np.sort(events[feature].to_numpy())
    thresholds = (values[:-1] + values[1:]) / 2

    for threshold in thresholds[1:-1]:  # Крайние пороги отрезают одну сработку
        variant_rows.append((feature, threshold))

variants = pd.DataFrame(
    data=variant_rows,
    columns=['feature', 'threshold'],
    index=pd.RangeIndex(1, len(variant_rows) + 1, name='variant'),
)

variants

Вторая подготовка — для задания 4: набор «две луны» (англ. *moons*), 400 точек двух классов с шумом, поровну разделённых на обучающий и валидационный наборы:

- `X_moons_train`, `y_moons_train` — 200 точек для обучения, признаки — матрица формы (200, 2);
- `X_moons_val`, `y_moons_val` — 200 точек для валидации.

In [ ]:
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split

X_moons, y_moons = make_moons(n_samples=400, noise=0.25, random_state=42)
X_moons_train, X_moons_val, y_moons_train, y_moons_val = train_test_split(
    X_moons,
    y_moons,
    test_size=0.5,
    random_state=42,
)

X_moons_train.shape, X_moons_val.shape

## Задание 1 (базовое, 1 балл)

Обучение дерева начинается с корня: алгоритм CART перебирает вопросы «признак ≤ порог?» и выбирает тот, после которого две части получаются самыми чистыми. Посчитайте вручную, на листке, насколько хорош вопрос вашего варианта.

**Дано:** таблица `events` и диаграмма сработок, ваш вопрос — строка таблицы `variants`.

**Получить** — на листке и затем в ячейке ниже:

- `variant` — номер вашего варианта;
- `gini_root` — неоднородность Джини (англ. *Gini impurity*) всех 15 сработок, то есть корня дерева;
- `left_people`, `left_false` — сколько людей и ложных сработок в левой части (признак ≤ порога); `right_people`, `right_false` — то же в правой части;
- `gini_left`, `gini_right` — неоднородность Джини левой и правой частей;
- `cost` — критерий CART $J(k, t_k)$ вашего вопроса.

**Как действовать:**

1. Найдите свой вариант в `variants`. Выпишите на листке номера сработок, которые попадают в левую часть, и номера сработок правой части; посчитайте в каждой части людей и ложные сработки.
2. Посчитайте неоднородность Джини корня и обеих частей: $G = 1 - \sum_k p_k^2$, где $p_k$ — доля класса $k$ в части.
3. Посчитайте взвешенную среднюю неоднородность частей: $J = \frac{m_{\text{л}}}{m} G_{\text{л}} + \frac{m_{\text{п}}}{m} G_{\text{п}}$, где $m_{\text{л}}$ и $m_{\text{п}}$ — число сработок в частях, $m = 15$.
4. Перенесите результаты в ячейку ниже выражениями с листка, а не округлёнными числами: `gini_left = 1 - (2/5)**2 - (3/5)**2`, `cost = 5/15 * gini_left + 10/15 * gini_right`.

**Чего не делать:** не считайте части и доли в Python — задание про расчёт руками; листок вы покажете на защите, а проверите себя в задании 2.

**Что почитать:**

- лекция 06, слайды «Мешок с шарами: мера беспорядка», «Формула Джини по частям», «Считаем Джини вручную» и «Критерий CART: две части, одно число»; книга, стр. 225–226 (уравнение 6.1) и стр. 228 (уравнение 6.2);
- [Обучение дерева решений](https://ru.wikipedia.org/wiki/Обучение_дерева_решений) — Википедия;
- [Decision tree learning](https://en.wikipedia.org/wiki/Decision_tree_learning) — Википедия на английском, раздел Gini impurity;
- руководство scikit-learn: [Decision Trees](https://scikit-learn.org/1.3/modules/tree.html), раздел Mathematical formulation — та же формула в обозначениях библиотеки.

In [ ]:
variant = ...  # Номер варианта
gini_root = ...  # Выражение с листка

left_people, left_false = ..., ...
right_people, right_false = ..., ...
gini_left = ...
gini_right = ...
cost = ...

In [ ]:
part_counts = [left_people, left_false, right_people, right_false]
assert variant in variants.index, 'Номер варианта — от 1 до 24'
assert all(isinstance(count, int) for count in part_counts), 'Число сработок в части — целое число'
assert sum(part_counts) == len(events), 'Левая и правая части вместе — все 15 сработок'
assert left_people + right_people == y_events.sum(), 'Людей в двух частях вместе столько же, сколько во всём наборе'
assert all(0 <= value <= 0.5 for value in (gini_root, gini_left, gini_right)), 'Джини двух классов — от 0 до 0,5'
assert min(gini_left, gini_right) <= cost <= max(gini_left, gini_right), 'J лежит между Джини частей'
assert cost < gini_root, 'Вопрос уменьшает неоднородность: J должен быть меньше Джини корня'

'Задание 1: самопроверка пройдена'

Вопрос: на сколько ваш вопрос уменьшил неоднородность по сравнению с корнем? Какая из частей чище и к Джини какой части ближе $J$ — почему?

*Ваш ответ:* …

## Задание 2 (базовое, 1 балл)

Проверьте листок деревом scikit-learn. Если обучить `DecisionTreeClassifier` на исходных признаках, он сам выберет лучший вопрос — скорее всего, не ваш. Поэтому дайте дереву единственный признак — ответ на ваш вопрос: разрезать данные такое дерево может только одним способом, и в его узлах окажутся ваши части.

**Дано:** `events`, `y_events`, `variants`; номер варианта `variant` и числа с листка из задания 1. Признак и порог вашего варианта заготовка кладёт в `my_feature` и `my_threshold`.

**Получить:**

- `question` — бинарный признак, таблица или матрица формы (15, 1): 0 — сработка уходит в левую часть (признак ≤ порога), 1 — в правую;
- `question_tree` — дерево глубины 1, обученное на `question` и `y_events` с `random_state=42`;
- `gini_root_sklearn`, `gini_left_sklearn`, `gini_right_sklearn` — неоднородность Джини корня, левого и правого потомка, взятые из обученного дерева;
- `cost_sklearn` — $J$, собранный из неоднородности и числа сработок потомков дерева.

**Как действовать:**

1. Постройте `question` сравнением столбца признака с порогом и переведите результат в 0 и 1.
2. Обучите дерево глубины 1 на `question`.
3. Обученное дерево хранит узлы в атрибуте `tree_`: номера потомков корня (корень — узел 0) — в `children_left` и `children_right`, неоднородность узлов — в `impurity`, число объектов — в `n_node_samples`. Возьмите оттуда числа корня и двух потомков и соберите $J$.
4. После самопроверки выполните ячейку с рисунком дерева и сравните узлы со своим листком.

**Чего не делать:** не вписывайте числа с листка в переменные `*_sklearn` — их должно посчитать дерево.

**Что почитать:**

- лекция 06, слайды «Что дерево помнит об узле» и «Джини появляется в узлах дерева»; книга, стр. 224–226 — атрибуты узла `gini`, `samples`, `value`;
- [Дерево решений](https://ru.wikipedia.org/wiki/Дерево_решений) — Википедия;
- документация: [`DecisionTreeClassifier`](https://scikit-learn.org/1.3/modules/generated/sklearn.tree.DecisionTreeClassifier.html) (атрибут `tree_`), [`plot_tree`](https://scikit-learn.org/1.3/modules/generated/sklearn.tree.plot_tree.html);
- пример scikit-learn: [Understanding the decision tree structure](https://scikit-learn.org/1.3/auto_examples/tree/plot_unveil_tree_structure.html) — как читать `children_left`, `children_right` и `impurity`.

In [ ]:
my_feature, my_threshold = variants.loc[variant, 'feature'], variants.loc[variant, 'threshold']

question = ...  # Ваш код
question_tree = ...

gini_root_sklearn = ...
gini_left_sklearn = ...
gini_right_sklearn = ...
cost_sklearn = ...

In [ ]:
tree_nodes = question_tree.tree_
assert np.shape(question) == (len(events), 1), 'question — таблица или матрица формы (15, 1)'
assert tree_nodes.node_count == 3, 'Дерево глубины 1 — это корень и два потомка'
assert tree_nodes.n_node_samples[tree_nodes.children_left[0]] == left_people + left_false, (
    'В левом потомке дерева не столько сработок, сколько в левой части листка: проверьте листок и направление сравнения'
)
assert np.isclose(gini_root_sklearn, gini_root), 'Джини корня из дерева не совпал с листком'
assert np.isclose(gini_left_sklearn, gini_left), 'Джини левой части из дерева не совпал с листком'
assert np.isclose(gini_right_sklearn, gini_right), 'Джини правой части из дерева не совпал с листком'
assert np.isclose(cost_sklearn, cost), 'J из дерева не совпал с листком'

'Задание 2: самопроверка пройдена'

In [ ]:
fig, axis = plt.subplots(figsize=(8, 4))
plot_tree(
    question_tree,
    feature_names=[f'{my_feature} > {my_threshold}'],
    class_names=['ложная', 'человек'],
    filled=True,
    ax=axis,
)
plt.show()

Как читать рисунок: в корне записано условие на бинарный признак, например `area > 9.5 <= 0.5` — «признак равен 0», то есть ответ на вопрос варианта «да», и такие сработки уходят влево; `value` — число ложных сработок и людей, в этом порядке.

Вопрос: почему дерево, обученное на одном бинарном признаке, задаёт именно ваш вопрос? Что изменится в дереве, если в `question` поставить 1 сработкам левой части, а не правой?

*Ваш ответ:* …

## Задание 3 (базовое, 1 балл)

В задании 1 вы проверили один вопрос, а группа — все 24. CART делает то же: считает $J$ каждого вопроса и ставит в корень вопрос с наименьшим $J$. Повторите этот шаг обучения кодом и сверьте результат с корнем дерева scikit-learn.

**Дано:** `events`, `y_events`, `X_events`, `variants`.

**Получить:**

- функцию `gini(labels)` — неоднородность Джини набора объектов по массиву их классов; классов может быть сколько угодно;
- функцию `split_cost(values, labels, threshold)` — критерий $J$ вопроса «значение ≤ порог» для значений признака `values` и классов `labels`;
- `costs` — `pd.Series`: индекс — номера вариантов из `variants`, значения — $J$;
- `best_variant` — номер варианта с наименьшим $J$;
- `root_tree` — дерево глубины 1 с `random_state=42`, обученное на `X_events` и `y_events`.

**Как действовать:**

1. Сначала запишите предсказание в ячейку ниже.
2. В `gini` посчитайте, сколько объектов каждого класса, переведите в доли и подставьте в формулу задания 1.
3. В `split_cost` разделите классы на две части маской «значение ≤ порог», посчитайте Джини частей и их взвешенную среднюю.
4. Пройдите по строкам `variants`, посчитайте $J$ каждого варианта и найдите вариант с минимумом.
5. Обучите `root_tree`. После самопроверки выполните готовые ячейки с графиком и рейтингом вариантов.

**Чего не делать:** в `gini` и `split_cost` не используйте scikit-learn — его дерево здесь только для сверки.

**Что почитать:**

- лекция 06, слайды «Перебираем пороги», «Лучший вопрос: минимум J» и «Алгоритм CART»; книга, стр. 228–229;
- [Жадный алгоритм](https://ru.wikipedia.org/wiki/Жадный_алгоритм) — Википедия: CART выбирает лучший вопрос только для текущего узла;
- документация: [`numpy.unique`](https://numpy.org/doc/1.24/reference/generated/numpy.unique.html) (параметр `return_counts`), [`pandas.Series.idxmin`](https://pandas.pydata.org/pandas-docs/version/2.0/reference/api/pandas.Series.idxmin.html), [`DecisionTreeClassifier`](https://scikit-learn.org/1.3/modules/generated/sklearn.tree.DecisionTreeClassifier.html).

Предсказание: не запуская код, посмотрите на диаграмму сработок. По какому признаку будет лучший вопрос и почему?

*Ваш ответ:* …

In [ ]:
def gini(labels):
    """Считает неоднородность Джини набора объектов.

    Args:
        labels: Классы объектов, массив NumPy или Series.

    Returns:
        Неоднородность Джини: 0 — все объекты одного класса.
    """
    ...  # Ваш код


def split_cost(values, labels, threshold):
    """Считает критерий CART J для вопроса «значение ≤ порог».

    Args:
        values: Значения признака.
        labels: Классы объектов.
        threshold: Порог вопроса.

    Returns:
        Взвешенная средняя неоднородность Джини левой и правой частей.
    """
    ...  # Ваш код


costs = ...  # Ваш код
best_variant = ...
root_tree = ...

In [ ]:
assert np.isclose(gini(np.array([1, 1, 0, 0])), 0.5), 'Два класса поровну — Джини 0,5'
assert np.isclose(gini(np.array([0, 0, 0])), 0.0), 'Объекты одного класса — Джини 0'
assert np.isclose(gini(np.array([0, 1, 2])), 2 / 3), 'Три класса поровну — Джини 2/3'
assert np.isclose(split_cost(np.array([1, 2, 3, 4]), np.array([0, 0, 1, 1]), 2.5), 0.0), 'Чистые части — J = 0'
assert len(costs) == len(variants) and (costs.index == variants.index).all(), 'Нужен J каждого из 24 вариантов'
assert np.isclose(costs[best_variant], costs.min()), 'best_variant — вариант с наименьшим J'
assert variants.loc[best_variant, 'feature'] == FEATURES[root_tree.tree_.feature[0]], (
    'Лучший вариант и корень дерева спрашивают о разных признаках'
)
assert np.isclose(variants.loc[best_variant, 'threshold'], root_tree.tree_.threshold[0]), (
    'Порог лучшего варианта не совпал с порогом корня дерева'
)

'Задание 3: самопроверка пройдена'

In [ ]:
fig, axis = plt.subplots(figsize=(8, 4))

for feature_name in FEATURES:
    is_feature = variants['feature'] == feature_name
    axis.plot(variants.loc[is_feature, 'threshold'], costs[is_feature], marker='o', label=feature_name)

axis.scatter(variants.loc[best_variant, 'threshold'], costs[best_variant], s=200, facecolors='none', edgecolors='black')
axis.axhline(gini(y_events), color='grey', linestyle='--', label='Джини корня')
axis.set_xlabel('Порог')
axis.set_ylabel('J')
axis.legend()
axis.grid(alpha=0.3)
plt.show()

In [ ]:
ranking = variants.assign(cost=costs.round(3)).sort_values('cost')
ranking['place'] = ranking['cost'].rank(method='min').astype(int)  # Равные J делят место

ranking

Вопрос: какое место у вашего варианта в рейтинге? Почему лучший вопрос по длительности заметно хуже лучшего вопроса по площади? Какой вопрос дерево задаст следующим в левой части после корня?

*Ваш ответ:* …

## Задание 4 (продвинутое, 3 балла)

Два слабых места деревьев из лекции. Первое — переобучение (англ. *overfitting*): без ограничений дерево делит данные до чистых листьев и заучивает шум. Его лечат гиперпараметрами регуляризации, например `min_samples_leaf` — минимум объектов в листе. Второе — неустойчивость: небольшое изменение данных меняет вопрос в корне, а за ним всё дерево.

**Дано:** `X_moons_train`, `y_moons_train`, `X_moons_val`, `y_moons_val` — луны из подготовки; `X_events`, `y_events`, `FEATURES` — сработки; `leaf_sizes` — значения `min_samples_leaf` от 1 до 30, массив уже записан в заготовке.

**Получить**, часть А — регуляризация на лунах, у всех деревьев `random_state=42`:

- `full_tree` — дерево без ограничений, обученное на обучающем наборе лун; `train_acc_full`, `val_acc_full` — его доля правильных ответов (англ. *accuracy*) на обучающем и валидационном наборах;
- `train_acc_by_leaf`, `val_acc_by_leaf` — массивы NumPy длины 30: доля правильных ответов деревьев с `min_samples_leaf` из `leaf_sizes`;
- `best_leaf_size` — значение `min_samples_leaf` с наибольшей долей на валидации; `best_tree` — дерево с этим значением.

Часть Б — неустойчивость на сработках:

- `full_root` — вопрос в корне дерева глубины 1 (`random_state=42`), обученного на всех 15 сработках: пара (имя признака, порог);
- `root_questions` — список из 15 таких пар: корень дерева, обученного на сработках без одной — по очереди без каждой;
- `changed_events` — список номеров сработок, без которых корень отличается от `full_root`.

**Как действовать:**

1. Сначала запишите предсказание в ячейку ниже.
2. Часть А: обучите `full_tree` и посчитайте обе доли; затем в цикле по `leaf_sizes` обучите по дереву и соберите доли; выберите лучшее значение по валидации.
3. Часть Б: напишите функцию, которая обучает дерево глубины 1 и возвращает пару (признак, порог) его корня, — пригодится атрибут `tree_` из задания 2. Вызовите её на всех сработках и в цикле по номерам сработок — на наборе без одной.
4. После самопроверок выполните готовые ячейки с графиками.

**Чего не делать:** не выбирайте `best_leaf_size` по обучающему набору — на нём всегда выигрывает дерево без ограничений.

**Что почитать:**

- лекция 06, слайды «Без ограничений дерево заучивает данные», «Ручки регуляризации», «Ограничение упрощает границу» и «Неустойчивость: одна машина меняет дерево»; книга, стр. 230–232 и 235–236;
- [Переобучение](https://ru.wikipedia.org/wiki/Переобучение) — Википедия;
- документация: [`make_moons`](https://scikit-learn.org/1.3/modules/generated/sklearn.datasets.make_moons.html), [`DecisionTreeClassifier`](https://scikit-learn.org/1.3/modules/generated/sklearn.tree.DecisionTreeClassifier.html) (параметр `min_samples_leaf`, методы `score` и `get_n_leaves`), [`accuracy_score`](https://scikit-learn.org/1.3/modules/generated/sklearn.metrics.accuracy_score.html), [`pandas.DataFrame.drop`](https://pandas.pydata.org/pandas-docs/version/2.0/reference/api/pandas.DataFrame.drop.html);
- руководство scikit-learn: [Decision Trees](https://scikit-learn.org/1.3/modules/tree.html), раздел Tips on practical use.

Предсказание: не запуская код, ответьте. (1) Какой будет доля правильных ответов дерева без ограничений на обучающем наборе лун и почему? (2) Что станет с долей на валидации, если `min_samples_leaf` очень большой — например, 30? (3) Сколько из 15 сработок, по-вашему, меняют корень дерева, если их убрать?

*Ваш ответ:* …

In [ ]:
leaf_sizes = np.arange(1, 31)

full_tree = ...  # Ваш код
train_acc_full, val_acc_full = ..., ...

train_acc_by_leaf, val_acc_by_leaf = ..., ...
best_leaf_size = ...
best_tree = ...

In [ ]:
assert train_acc_full > val_acc_full, 'Дерево без ограничений должно быть точнее на обучении, чем на валидации'
assert np.shape(train_acc_by_leaf) == leaf_sizes.shape == np.shape(val_acc_by_leaf), 'Нужна доля на каждое значение'
assert np.isclose(train_acc_by_leaf[0], train_acc_full), 'min_samples_leaf=1 — это то же дерево без ограничений'
assert train_acc_by_leaf[-1] < train_acc_by_leaf[0], 'Крупные листья должны хуже запоминать обучающий набор'
assert np.isclose(val_acc_by_leaf[best_leaf_size - 1], val_acc_by_leaf.max()), 'best_leaf_size выбирается по валидации'
assert best_tree.get_params()['min_samples_leaf'] == best_leaf_size, 'best_tree обучен не с best_leaf_size'
assert val_acc_by_leaf.max() > val_acc_full, 'Регуляризация должна улучшить долю правильных ответов на валидации'
assert best_tree.get_n_leaves() < full_tree.get_n_leaves(), 'У регуляризованного дерева должно быть меньше листьев'

'Задание 4, часть А: самопроверка пройдена'

In [ ]:
from sklearn.inspection import DecisionBoundaryDisplay

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].plot(leaf_sizes, train_acc_by_leaf, label='обучающий набор')
axes[0].plot(leaf_sizes, val_acc_by_leaf, label='валидационный набор')
axes[0].set_xlabel('min_samples_leaf')
axes[0].set_ylabel('Доля правильных ответов')
axes[0].legend()
axes[0].grid(alpha=0.3)

for axis, (title, model) in zip(axes[1:], [('без ограничений', full_tree), ('лучший min_samples_leaf', best_tree)]):
    DecisionBoundaryDisplay.from_estimator(model, X_moons_train, response_method='predict', alpha=0.3, ax=axis)
    axis.scatter(X_moons_train[:, 0], X_moons_train[:, 1], c=y_moons_train, s=12, edgecolors='white')
    axis.set_title(f'{title}: листьев {model.get_n_leaves()}')

plt.show()

In [ ]:
full_root = ...  # Ваш код
root_questions = ...
changed_events = ...

In [ ]:
assert len(full_root) == 2 and full_root[0] in FEATURES, 'full_root — пара (имя признака, порог)'
assert len(root_questions) == len(events), 'Нужен корень для каждой из 15 сработок'
assert all(len(root) == 2 and root[0] in FEATURES for root in root_questions), 'Корень — пара (признак, порог)'
assert set(changed_events) <= set(events.index), 'changed_events — номера сработок из events'
assert len(changed_events) == sum(root != full_root for root in root_questions), (
    'changed_events — ровно те сработки, без которых корень отличается от full_root'
)
assert 0 < len(changed_events) < len(events), 'Корень должен меняться без некоторых, но не без всех сработок'

'Задание 4, часть Б: самопроверка пройдена'

In [ ]:
events.assign(root_without_event=[f'{feature} ≤ {threshold}' for feature, threshold in root_questions])

Вопросы: (1) по своим графикам — при каком `min_samples_leaf` лучшая доля на валидации и почему при меньших и больших значениях она хуже? (2) Без каких сработок меняется корень и почему именно без них? (3) Все ли изменения корня — действительно другой вопрос? Сравните части, на которые делят сработки старый и новый вопросы.

*Ваш ответ:* …

## Готово к защите

Перед тем как звать преподавателя:

- перезапустите ядро (Kernel → Restart) и выполните сверху вниз подготовку данных и защищаемые задания;
- самопроверки защищаемых заданий проходят, ячейки с графиками и таблицами выполнены;
- для задания 1 — листок с расчётом своего варианта под рукой: на защите вас попросят посчитать на листке ещё один вариант;
- поля «*Ваш ответ:*» защищаемых заданий заполнены.